# Quickstart: 5 分钟跑通 DID Pipeline

> 演示从数据到 DID 系数的最小流程。使用 `data/sample/esg_panel_demo.csv` 合成 fixture。
> 完整版 (含真实 MCP 调用): `python scripts/agent_pipeline.py --topic "碳排放权交易 DID"`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Load sample fixture
DATA_PATH = Path('../data/sample/esg_panel_demo.csv')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} obs, {df.firm_id.nunique()} firms, {df.year.nunique()} years')
df.head()

In [ ]:
# 2. Initialize ModernDiDEngine
from scripts.research_framework.modern_did import ModernDiDEngine

engine = ModernDiDEngine(
    df=df,
    y_var='lev',
    treat_var='did',  # post × esg_high
    time_var='post',
    unit_var='firm_id',
)
print(f'Engine ready: y={engine.y_var}, treat={engine.treat_var}, time={engine.time_var}, unit={engine.unit_var}')

In [ ]:
# 3. Standard 2x2 DID with cluster-robust SE
result = engine.did_2x2(cluster_var='firm_id')
print(f'\n=== DID Coefficient ===')
print(f'  Coef: {result.coef:.4f} (true effect = -0.05)')
print(f'  SE:   {result.se:.4f}')
print(f'  p-value: {result.pval:.4f}')
print(f'  Sig:  {result.sig}')
print(f'  N:    {result.n_obs}')
print(f'  Method: {result.method}')

In [ ]:
# 4. Event Study (平行趋势检验)
try:
    es_result = engine.event_study()
    print(f'\n=== Event Study ===')
    print(f'  Pre-trend F-test p-value: {es_result.additional.get("pretrend_pval", "N/A")}')
except Exception as e:
    print(f'Event study failed (likely too few periods): {e}')
    print(f'  Try: more years in your fixture (>= 5 pre + 5 post recommended)')

In [ ]:
# 5. Save result to markdown table
from pathlib import Path
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

table_md = f"""| Variable | Coef | SE | p | Sig | N |
|---|---|---|---|---|---|
| DID | {result.coef:.4f} | ({result.se:.4f}) | {result.pval:.4f} | {result.sig} | {result.n_obs} |
"""
(out_dir / 'did_table.md').write_text(table_md)
print('✓ Saved to output/did_table.md')
print(table_md)

## 下一步

1. **替换数据**: 用 `user-tushare` (A 股) 或 `user-yfinance` (美股) 替换 `data/sample/esg_panel_demo.csv`
2. **加控制变量**: 传入 `x_vars=['roa', 'ln_assets', 'tangibility']` 到 `engine.did_2x2`
3. **跑稳健性**: 19 项自动稳健性检验 `RobustnessRunner.run_comprehensive('full')`
4. **生成 LaTeX**: `engine.save_latex_table('output/table3.tex')`

完整文档: `使用指南.md` (1048 行) · `docs/tutorials/01-quickstart.md`